In [ ]:
import pyproj
from geo_data import data_handler, helpers, svg_handler

In [ ]:
countries = data_handler.load("country", "ne", resolution=110)

In [ ]:
# center of europe
europe = countries[countries["continent"] == "Europe"]

exclude = ['RUS', 'ISL', 'TUR']  # exclude Russia, Iceland, Turkey
europe = europe[~europe['iso_a3'].isin(exclude)]

center = europe.union_all().centroid

In [ ]:
# orthographic projection
ortho_proj_str = f"+proj=ortho +lat_0={center.y} +lon_0={center.x}"
ortho_proj = pyproj.CRS.from_proj4(ortho_proj_str)

# project the countries
world_ortho = countries.to_crs(ortho_proj)

# Filter out polygons with inf coordinates
world_ortho = world_ortho[~world_ortho["geometry"].apply(data_handler.polygon_is_all_inf)]
world_ortho = world_ortho.sort_values("admin", ascending=False)

In [ ]:
svg_size = 1000
world_radius = ortho_proj.ellipsoid.semi_major_metre

file_path = helpers.get_top_directory() / "results" / "europe_orthographic.svg"
dwg = svg_handler.MapSVG(file_path, size=svg_size)

dwg.add_circle(
    "sea",
    center=(svg_size / 2, svg_size / 2),
    r=svg_size / 2,
    **dwg.get_kwargs("sea")
)

dwg.add_group("countries", **dwg.get_kwargs("land"))
for i, country in world_ortho.iterrows():
    geom = country.geometry
    country_id = country["admin"].replace(" ", "_")
    dwg.add_polygon(geom, x_lim=(-world_radius, world_radius), polygon_id=country_id, group_id="countries")
dwg.save()
dwg.drawing

# TODO: add gradient for 3D effect